## `funding_curated` gap backfill (one-off, standalone)

Two pre-existing gaps in `funding_curated`, both now fixed at the source for every *future* row
(`S2_grant_deduplication.ipynb`'s `gt_lrd_col_map` casing bug for Grants Tracker rows;
`S6_LLM_labelling.py`'s promotion block for Dimensions rows) but still missing on rows already
imported before those fixes existed:

1. **Grants-Tracker-origin rows** - `'End date'` / `'duration (years)'` / `'Years active'` were
   never actually imported (a case-typo in the column mapping silently no-op'd every run).
2. **Dimensions-origin rows** - never got a `'Gov contribution'` / `'Gov contribution (USD)'` /
   `'Gov & NP contribution (EUR)'` figure (Dimensions grants are fully government-funded, so this
   should just equal the total amount), and mostly never got `'duration (years)'` / `'Years
   active'` either despite having both a start and end year.

Every write below only fills a currently-blank cell via `COALESCE` - nothing here overwrites an
existing value. One-off - safe to delete once run.

In [1]:
import re
from datetime import datetime

import duckdb
import pandas as pd
from pathlib import Path

from Funding_dedup_helpers import assign_stable_row_id, extract_year, derive_duration_and_years_active

DB_PATH = Path('funding.db')
RAW_DATA_DIR = Path('raw_data')
OUTPUT_DIR = Path('data_output')
OUTPUT_DIR.mkdir(exist_ok=True)
GRANTS_TRACKER_FILE = 'GrantsTracker_2026-06-30_edit-start-year.xlsx'  # same file S2 last used
MANUAL_EDITS_FILE = '260727_0922_ap_pillar_backfill_funding_curated_MANUAL-EDITS.xlsx'

# Captured once so every output from this run shares one timestamp, however long the run takes.
RUN_TIMESTAMP = datetime.today().strftime('%y%m%d_%H%M')

_GRANT_ID_RE = re.compile(r'^grant\.\d+$')

### 1. Backfill Grants-Tracker-origin rows

In [2]:
gt = pd.read_excel(RAW_DATA_DIR / GRANTS_TRACKER_FILE)
gt = assign_stable_row_id(gt, 'gt_row_id')  # stamped before any filtering, matching S2's convention

gt['Year project started'] = pd.to_numeric(gt['EXT_Year project starts'], errors='coerce')
gt['_end_year_derived'] = gt['INT_End Date'].apply(extract_year)
_derived = gt.apply(
    lambda r: derive_duration_and_years_active(r['Year project started'], r['_end_year_derived']),
    axis=1,
)
gt['duration (years)'] = _derived.apply(lambda t: t[0])
gt['Years active'] = _derived.apply(lambda t: t[1])
gt['End date'] = gt['_end_year_derived']
gt['grant_id'] = gt['Dimensions.ai grant ID'].astype(str).str.strip()

PATCH_COLS = ['End date', 'duration (years)', 'Years active', 'Year project started']

# 'EXT_Year project starts' was ALSO mistyped as 'EXT_Years project starts' in gt_lrd_col_map
# before this fix (a second, separate instance of the same casing/naming bug as 'INT_End date') -
# checked live: 441 airtable-origin rows are missing 'Year project started', 169 of which are
# recoverable from this raw file. Included here alongside the end-date/duration backfill since
# it's the exact same matching logic.
gt_patch = gt[gt['grant_id'].str.match(_GRANT_ID_RE) & (gt['End date'].notna() | gt['Year project started'].notna())][
    ['grant_id', 'gt_row_id'] + PATCH_COLS
].copy()
gt_patch['lrd_row_id_key'] = 'GT_' + gt_patch['gt_row_id'].astype(str)
print(f"{len(gt_patch)} Grants Tracker rows have a usable derived end date and/or start year to backfill with.")

con = duckdb.connect(str(DB_PATH))

set_clause = ', '.join(f'"{col}" = COALESCE(funding_curated."{col}", {{table}}."{col}")' for col in PATCH_COLS)

# Rows appended as brand-new funding_curated rows (never matched to an existing last-report row) -
# identified by the synthetic 'GT_<gt_row_id>' key S2 stamps on them.
con.register('gt_patch_append', gt_patch[['lrd_row_id_key'] + PATCH_COLS])
con.sql(f'''
    UPDATE funding_curated
    SET {set_clause.format(table='gt_patch_append')}
    FROM gt_patch_append
    WHERE funding_curated.lrd_row_id = gt_patch_append.lrd_row_id_key
''')
con.unregister('gt_patch_append')

# Rows that were gap-filled into an existing last-report row by a shared Dimensions grant ID -
# match by 'Identification code' first, falling back to 'Legacy identification code' for rows
# whose ID was later superseded by a Dimensions title-match (REVIEW POINT 3/4's overwrite logic).
con.register('gt_patch_id', gt_patch[['grant_id'] + PATCH_COLS])
for id_col in ['Identification code', 'Legacy identification code']:
    con.sql(f'''
        UPDATE funding_curated
        SET {set_clause.format(table='gt_patch_id')}
        FROM gt_patch_id
        WHERE funding_curated."{id_col}" = gt_patch_id.grant_id
    ''')
con.unregister('gt_patch_id')

n_now = con.sql('''
    SELECT
        SUM(CASE WHEN "duration (years)" IS NOT NULL THEN 1 ELSE 0 END) AS has_dur,
        SUM(CASE WHEN "Year project started" IS NOT NULL THEN 1 ELSE 0 END) AS has_start,
        COUNT(*) AS n
    FROM funding_curated WHERE "Database" = 'airtable'
''').df()
print(n_now.to_string())

907 Grants Tracker rows have a usable derived end date and/or start year to backfill with.
   has_dur  has_start     n
0    841.0      911.0  1183


### 2. Backfill Dimensions-origin rows

In [ ]:
# Gov-contribution copy: when nothing more precise is known, assume the total amount IS the gov
# contribution. Applied universally (not gated by 'Database') - checked live: 12 rows tagged with
# an older/different Grants-Tracker label (e.g. 'UKRI', 'airtable 2025') are genuinely GT-linked
# (their Identification code matches the raw GT file) and would be missed by a 'Database'-based
# filter, so this applies to any row with the same total-but-no-gov gap regardless of source.
# Purely row-local (no join needed).
con.sql('''
    UPDATE funding_curated
    SET
        "Gov contribution" = COALESCE("Gov contribution", "Total amount"),
        "Gov contribution (USD)" = COALESCE("Gov contribution (USD)", "Total amount (USD)"),
        "Gov & NP contribution (EUR)" = COALESCE("Gov & NP contribution (EUR)", "Total amount (EUR)")
''')

# duration (years) / Years active derivation stays Dimensions-only - Grants-Tracker rows already
# get this from Section 1's GT-matching backfill above.
dim_df = con.sql('''
    SELECT "Identification code", "Year project started", "End date"
    FROM funding_curated
    WHERE "Database" = 'Dimensions'
''').df()

_derived_dim = dim_df.apply(
    lambda r: derive_duration_and_years_active(r['Year project started'], r['End date']),
    axis=1,
)
dim_df['duration (years)'] = _derived_dim.apply(lambda t: t[0])
dim_df['Years active'] = _derived_dim.apply(lambda t: t[1])

con.register('dim_patch', dim_df)
con.sql('''
    UPDATE funding_curated
    SET
        "duration (years)" = COALESCE(funding_curated."duration (years)", dim_patch."duration (years)"),
        "Years active" = COALESCE(funding_curated."Years active", dim_patch."Years active")
    FROM dim_patch
    WHERE funding_curated."Identification code" = dim_patch."Identification code"
    AND funding_curated."Database" = 'Dimensions'
''')
con.unregister('dim_patch')

summary = con.sql('''
    SELECT
        SUM(CASE WHEN "Total amount" IS NOT NULL THEN 1 ELSE 0 END) AS has_total,
        SUM(CASE WHEN "Gov contribution" IS NOT NULL THEN 1 ELSE 0 END) AS has_gov,
        SUM(CASE WHEN "Gov & NP contribution (EUR)" IS NOT NULL THEN 1 ELSE 0 END) AS has_gov_eur,
        COUNT(*) AS n
    FROM funding_curated
''').df()
print("Gov-contribution copy (whole table):")
print(summary.to_string())

dim_summary = con.sql('''
    SELECT
        SUM(CASE WHEN "duration (years)" IS NOT NULL THEN 1 ELSE 0 END) AS has_dur,
        COUNT(*) AS n
    FROM funding_curated WHERE "Database" = 'Dimensions'
''').df()
print("Dimensions duration (years):")
print(dim_summary.to_string())

In [4]:
# Excel copy of the backfilled funding_curated dataset, for a before/after record of this one-off fix.
export_path = OUTPUT_DIR / f'{RUN_TIMESTAMP}_end-date-gap-backfill.xlsx'
con.sql('SELECT * FROM funding_curated').df().to_excel(export_path, index=False)
print(f"Excel copy of 'funding_curated' saved to {export_path}")

con.close()

Excel copy of 'funding_curated' saved to data_output\260804_1608_end-date-gap-backfill.xlsx


### 3. Apply the same backfill to the manually-edited Excel export

Independent of Sections 1-2 above (which write into `funding.db`) - this applies the exact same
derivation logic to `MANUAL_EDITS_FILE`, matched by `lrd_row_id`/`'Identification code'` rather than
row position (rows have been added/removed/split by hand since that file was exported), and only
fills currently-blank cells so nothing already hand-edited gets touched. Saved as a new file -
the original manually-edited file is left untouched.

In [ ]:
manual_df = pd.read_excel(OUTPUT_DIR / MANUAL_EDITS_FILE)
print(f"Loaded {len(manual_df)} rows from '{MANUAL_EDITS_FILE}'.")

# --- Grants-Tracker-origin rows: reuse the same gt_patch computed in Section 1 above ---
gt_patch_unique_lrd = gt_patch.drop_duplicates(subset=['lrd_row_id_key'])
gt_patch_unique_id = gt_patch.drop_duplicates(subset=['grant_id'])


def fillna_from_gt_patch(df, on_col, patch, patch_key_col):
    """Left-merge patch's PATCH_COLS onto df (matched on df[on_col] == patch[patch_key_col]),
    then fillna df's own PATCH_COLS from the matched values - never overwrites a non-blank cell.
    Both halves of a manually-split row share the same lrd_row_id, so both correctly receive the
    same derived end date/duration/years-active (same underlying Grants-Tracker timeline)."""
    renamed = patch[[patch_key_col] + PATCH_COLS].rename(columns={c: c + '_gt' for c in PATCH_COLS})
    merged = df.merge(renamed, how='left', left_on=on_col, right_on=patch_key_col)
    assert len(merged) == len(df), "merge produced extra rows - a patch key wasn't unique"
    for col in PATCH_COLS:
        df[col] = df[col].fillna(pd.Series(merged[col + '_gt'].values, index=df.index))
    return df


n_before = manual_df['duration (years)'].notna().sum()
manual_df = fillna_from_gt_patch(manual_df, 'lrd_row_id', gt_patch_unique_lrd, 'lrd_row_id_key')
manual_df = fillna_from_gt_patch(manual_df, 'Identification code', gt_patch_unique_id, 'grant_id')
manual_df = fillna_from_gt_patch(manual_df, 'Legacy identification code', gt_patch_unique_id, 'grant_id')
print(f"'duration (years)' populated: {n_before} -> {manual_df['duration (years)'].notna().sum()} (of {len(manual_df)} rows)")

# --- Gov-contribution copy: universal (not gated by Database) - see Section 2's comment above
# for why this isn't scoped to a specific Database label.
manual_df['Gov contribution'] = manual_df['Gov contribution'].fillna(manual_df['Total amount'])
manual_df['Gov contribution (USD)'] = manual_df['Gov contribution (USD)'].fillna(manual_df['Total amount (USD)'])
manual_df['Gov & NP contribution (EUR)'] = manual_df['Gov & NP contribution (EUR)'].fillna(manual_df['Total amount (EUR)'])

# --- Dimensions-origin rows: duration/Years active derivation stays Dimensions-only, same as
# Section 2 above - Grants-Tracker rows already got this from the fillna_from_gt_patch calls above.
dim_mask = manual_df['Database'] == 'Dimensions'
_derived_dim = manual_df.loc[dim_mask].apply(
    lambda r: derive_duration_and_years_active(r['Year project started'], r['End date']),
    axis=1,
)
manual_df.loc[dim_mask, 'duration (years)'] = manual_df.loc[dim_mask, 'duration (years)'].fillna(_derived_dim.apply(lambda t: t[0]))
manual_df.loc[dim_mask, 'Years active'] = manual_df.loc[dim_mask, 'Years active'].fillna(_derived_dim.apply(lambda t: t[1]))

export_path = OUTPUT_DIR / MANUAL_EDITS_FILE.replace('.xlsx', '_backfilled.xlsx')
manual_df.to_excel(export_path, index=False)
print(f"Backfilled copy saved to {export_path} - original '{MANUAL_EDITS_FILE}' left untouched.")